In [1]:
import csv
import numpy as np
import pandas as pd
import sys, os
import re
import ast
import datetime as dt
from tqdm import tqdm
from collections import defaultdict, Counter

import warnings

# 禁用所有警告
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
# df = pd.read_csv('D:/2024BI_Journal/MIMICIV_31_cohort_icu_mortality.csv')
# df.shape
df = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/icustays.csv.gz')
df = df[df.los>=1]
df.shape

(74829, 8)

In [ ]:
# ['patientweight']

In [3]:
ori = ['PROC_221214', 'PROC_221216', 'PROC_221217', 'PROC_221223', 'PROC_221255', 'PROC_223253', 'PROC_224263', 'PROC_224264', 'PROC_224267', 'PROC_224270', 'PROC_224385', 'PROC_225400', 'PROC_225401', 'PROC_225402', 'PROC_225432', 'PROC_225441', 'PROC_225444', 'PROC_225451', 'PROC_225454', 'PROC_225462', 'PROC_225752', 'PROC_225792', 'PROC_225794', 'PROC_225802', 'PROC_225814', 'PROC_225817', 'PROC_225966', 'PROC_227194', 'PROC_229351', 'MEDS_221289', 'MEDS_221468', 'MEDS_221555', 'MEDS_221653', 'MEDS_221662', 'MEDS_221744', 'MEDS_221749', 'MEDS_221794', 'MEDS_221906', 'MEDS_221986', 'MEDS_222042', 'MEDS_222056', 'MEDS_222168', 'MEDS_222315', 'MEDS_223258', 'MEDS_225152', 'MEDS_225168', 'MEDS_225170', 'MEDS_229630', 'OUT_226571', 'OUT_226579', 'OUT_226627']
ids = []
for i in ori:
    ids.append(int(i.split('_')[1]))

In [4]:
d_items = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/d_items.csv.gz')

In [5]:
proc = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/procedureevents.csv.gz')

In [6]:
proc=proc[['stay_id','itemid','starttime', 'endtime', 'value', 'ordercategoryname']]
proc=proc[proc['stay_id'].isin(df['stay_id'].unique())]

In [7]:
proc.head(2)

,stay_id,itemid,starttime,endtime,value,ordercategoryname
3,37081114,224275,2150-11-02 20:00:00,2150-11-03 13:00:00,1020.0,Peripheral Lines
4,37081114,224277,2150-11-02 20:38:00,2150-11-03 11:59:00,921.0,Peripheral Lines


In [14]:
print(list(set(ids)-set(proc.itemid.unique())))

[225152, 226571, 225168, 225170, 226579, 223258, 221468, 221986, 221744, 221749, 226627, 221906, 221653, 222168, 222042, 221662, 221794, 222056, 221289, 222315, 221555, 229630]


In [15]:
print(list(set(proc.itemid.unique())&set(ids)))

[225792, 224385, 225794, 224263, 224264, 225802, 224267, 224270, 223253, 225814, 225432, 225817, 221214, 221216, 221217, 225441, 225444, 221223, 225451, 225454, 225966, 225462, 221255, 225752, 225402, 229351, 225400, 225401, 227194]


In [8]:
procs = [225401, 225437, 225444, 225451, 225454, 225722, 225723, 225724,
225725, 225726, 225727, 225728, 225729, 225730, 225731, 225732,
225733, 225734, 225735, 225736, 225768, 225814, 225816, 225817,
225818, 226131, 227726,225399,225400,225401,225427,225430,225433,225434,225439,225440,225441,225442,225445,225448,225449,225450,225464,225464,225466,225467,
225472,225475,225479,225752,225792,225794,225802,225805,225966,225967,226237,227194,227712,227713,228715,229351,
224385,221214,221216,221217,221223,223253,224264,224268,224270,224272,224273,225402]
procs = pd.unique(procs)
len(procs)

72

In [9]:
proc=proc[proc['itemid'].isin(procs)]
proc=proc.dropna(subset=['value'])

In [10]:
proc.shape

(335585, 6)

In [11]:
len(proc.itemid.unique())

53

In [12]:
proc.head(2)

,stay_id,itemid,starttime,endtime,value,ordercategoryname
6,37081114,225402,2150-11-05 12:00:00,2150-11-05 12:01:00,1.0,Procedures
7,37081114,225401,2150-11-05 17:21:00,2150-11-05 17:22:00,1.0,Procedures


In [13]:
proc = pd.merge(proc,df[['stay_id', 'intime', 'outtime']],on='stay_id',how='left')

In [14]:
proc['intime'] = pd.to_datetime(proc['intime'])
proc['outtime'] = pd.to_datetime(proc['outtime'])
proc['starttime'] = pd.to_datetime(proc['starttime'])
proc['endtime'] = pd.to_datetime(proc['endtime'])

In [15]:
d_items[d_items.itemid.isin(proc.itemid.unique())]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
272,221214,CT scan,CT scan,procedureevents,5-Imaging,None,Processes,NaN,NaN
273,221216,X-ray,X-ray,procedureevents,5-Imaging,None,Processes,NaN,NaN
274,221217,Ultrasound,Ultrasound,procedureevents,5-Imaging,None,Processes,NaN,NaN
276,221223,EEG,EEG,procedureevents,4-Procedures,None,Processes,NaN,NaN
322,223253,Magnetic Resonance Imaging,MRI,procedureevents,5-Imaging,None,Processes,NaN,NaN
598,224264,PICC Line,PICC Line,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN
600,224268,Trauma line,Trauma line,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN
602,224270,Dialysis Catheter,Dialysis Catheter,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN
603,224272,IABP line,IABP line,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN
604,224273,Presep Catheter,Presep Catheter,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN


In [16]:
within_criteria = proc['starttime'].between(proc['intime'],proc['outtime'])
within = proc[within_criteria]

In [17]:
within['event_time_from_admit'] = within['starttime'] - within['intime']
within['start_time']=within['event_time_from_admit'].dt.total_seconds() / 3600
within['los'] = (within['outtime'] - within['intime']).dt.total_seconds() / 3600
within=within[within['start_time']>=0]
within.head(2)

,stay_id,itemid,starttime,endtime,value,ordercategoryname,intime,outtime,event_time_from_admit,start_time,los
0,37081114,225402,2150-11-05 12:00:00,2150-11-05 12:01:00,1.0,Procedures,2150-11-02 19:37:00,2150-11-06 17:03:17,2 days 16:23:00,64.383333,93.438056
1,37081114,225401,2150-11-05 17:21:00,2150-11-05 17:22:00,1.0,Procedures,2150-11-02 19:37:00,2150-11-06 17:03:17,2 days 21:44:00,69.733333,93.438056


In [18]:
within_72 = within[within['start_time']<=72]
within_72.shape

(235457, 11)

In [19]:
# 定义一个函数来判断值类型
def is_numeric(value):
    try:
        float(value)  # 尝试将值转换为浮点数
        return True
    except (ValueError, TypeError):
        return False

# 添加一列标记 value 是否为数值
within_72['is_numeric'] = within_72['value'].apply(is_numeric)

# 按 itemid 判断是否所有值都为数值
result = within_72.groupby('itemid')['is_numeric'].all().reset_index()
result.rename(columns={'is_numeric': 'is_fully_numeric'}, inplace=True)

num_col = result[result.is_fully_numeric==True].itemid.values
str_col = result[result.is_fully_numeric==False].itemid.values

In [20]:
num_col

array([221214, 221216, 221217, 221223, 223253, 224264, 224268, 224270,
       224272, 224273, 224385, 225399, 225400, 225401, 225402, 225427,
       225430, 225433, 225434, 225437, 225439, 225440, 225441, 225442,
       225444, 225445, 225448, 225449, 225450, 225451, 225454, 225464,
       225466, 225467, 225472, 225475, 225479, 225752, 225792, 225794,
       225802, 225805, 225814, 225816, 225817, 225966, 225967, 226237,
       227194, 227712, 227713, 228715, 229351], dtype=int64)

In [21]:
str_col

array([], dtype=int64)

In [22]:
def compute_outlier_imputation(arr, upper_percentile, lower_percentile):
    """
    按给定分位数对数组进行异常值替换。
    Args:
        arr (array-like): 输入的数值数组。
        upper_percentile (float): 上百分位数阈值。
        lower_percentile (float): 下百分位数阈值。
    Returns:
        np.ndarray: 处理后的数组。
    """
    arr = pd.to_numeric(arr, errors='coerce')
    # 计算上下限阈值
    upper_bound = np.percentile(arr, upper_percentile)
    lower_bound = np.percentile(arr, lower_percentile)
    
    # 替换异常值
    arr[arr < lower_bound] = lower_bound
    arr[arr > upper_bound] = upper_bound

    return arr

def outlier_imputation(data, id_attribute, value_attribute, num_col, upper_percentile=98, lower_percentile=2):
    """
    对指定列按分组进行异常值替换，仅对 num_col 中的 id_attribute 进行操作。
    Args:
        data (DataFrame): 输入数据。
        id_attribute (str): 分组的列名。
        value_attribute (str): 要处理的列名。
        num_col (list): 需要处理的 id_attribute 的值。
        upper_percentile (float): 上百分位数阈值，默认98。
        lower_percentile (float): 下百分位数阈值，默认2。
    Returns:
        DataFrame: 处理后的 DataFrame。
    """
    # 按指定的分组列进行分组
    grouped = data.groupby(id_attribute)

    # 遍历每个分组
    for id_number, group in grouped:
        # 仅处理 num_col 中的 id_attribute
        if id_number not in num_col:
            continue
        
        index = group.index
        values = group[value_attribute].values

        # 调用异常值处理函数
        imputed_values = compute_outlier_imputation(values, upper_percentile, lower_percentile)
        
        # 替换处理后的值
        data.loc[index, value_attribute] = imputed_values

    # 删除因异常值处理引入的 NaN
    data = data.dropna(subset=[value_attribute])

    return data

In [23]:
within_72 = outlier_imputation(within_72, 'itemid', 'value', num_col, upper_percentile=98, lower_percentile=2)

In [24]:
final_chart=pd.DataFrame()
t=0

for i in tqdm(np.arange(0, 72, 0.25)):  # 时间间隔调整为每15分钟,72*4=288
    # 根据时间段筛选数据并按 'stay_id' 和 'itemid' 分组，计算均值
    sub_chart = within_72[(within_72['start_time'] >= i) & (within_72['start_time'] < i + 0.25)]
    
    # 对于数值型 itemid，计算均值或中位数
    sub_chart_num = sub_chart[sub_chart['itemid'].isin(num_col)].groupby(['stay_id', 'itemid']).agg({'value': np.nanmean}).reset_index()

    # 对于字符串型 itemid，将所有值组合为一个列表
    sub_chart_str = sub_chart[sub_chart['itemid'].isin(str_col)].groupby(['stay_id', 'itemid']).agg({'value': lambda x: list(x)}).reset_index()

    # 合并数值型和字符串型的数据
    sub_chart = pd.concat([sub_chart_num, sub_chart_str], ignore_index=True)

    # 添加时间信息
    sub_chart['start_time'] = t

    # 将结果添加到 final_chart
    if final_chart.empty:
        final_chart = sub_chart
    else:
        final_chart = final_chart.append(sub_chart)

    t += 1

100%|████████████████████████████████████████████████████████████████████████████████| 288/288 [00:12<00:00, 22.77it/s]


In [25]:
final_chart.shape

(233445, 4)

In [26]:
len(final_chart.stay_id.unique())

57475

In [27]:
final_chart.head(2)

,stay_id,itemid,value,start_time
0,30000646,225402,1.0,0
1,30003306,225752,125.0,0


In [28]:
cate_01 = []
for it in final_chart.itemid.unique():
    if len(final_chart[final_chart.itemid==it].value.unique())>2:
        print(it,d_items[d_items.itemid==it].label.values)
        print(final_chart[final_chart.itemid==it].value.value_counts())
    else:
        cate_01.append(it)

225752 ['Arterial Line']
18378.000000    608
2.944444        600
1260.000000      46
1140.000000      45
1200.000000      41
               ... 
17896.000000      1
17040.000000      1
3882.000000       1
6410.000000       1
8455.000000       1
Name: value, Length: 9493, dtype: int64
225792 ['Invasive Ventilation']
26368.98    547
107.00      546
210.00       62
240.00       58
300.00       57
           ... 
7904.00       1
21598.00      1
7280.00       1
4900.00       1
7215.00       1
Name: value, Length: 9004, dtype: int64
229351 ['Foley Catheter']
2.513806        327
22165.760000    327
1200.000000      19
1440.000000      17
900.000000       17
               ... 
7553.000000       1
194.000000        1
5817.000000       1
8702.000000       1
9419.000000       1
Name: value, Length: 7511, dtype: int64
224264 ['PICC Line']
27499.880000    186
4.965583        185
1586.000000       9
1441.000000       8
1650.000000       7
               ... 
4339.000000       1
838.000000        1


In [29]:
print(cate_01,len(cate_01))

[225402, 225966, 225454, 225401, 225967, 224385, 221216, 221214, 225433, 225449, 225430, 225400, 225464, 225427, 221217, 225451, 225466, 221223, 225445, 225475, 225479, 225399, 225444, 225439, 225814, 223253, 225816, 225448, 227194, 225440, 227712, 225434, 225817, 225437, 227713, 225472, 225450, 225467, 226237, 228715, 225442] 41


In [30]:
for it in cate_01:
    print(it,d_items[d_items.itemid==it].label.values)
    print(final_chart[final_chart.itemid==it].value.value_counts())

225402 ['EKG']
1.0    30607
Name: value, dtype: int64
225966 ['Nasal Swab']
1.0    9418
Name: value, dtype: int64
225454 ['Urine Culture']
1.0    7309
Name: value, dtype: int64
225401 ['Blood Cultured']
1.0    13153
Name: value, dtype: int64
225967 ['Rectal Swab']
1.0    72
Name: value, dtype: int64
224385 ['Intubation']
1.0    6639
Name: value, dtype: int64
221216 ['X-ray']
1.0    4047
Name: value, dtype: int64
221214 ['CT scan']
1.0    15201
Name: value, dtype: int64
225433 ['Chest Tube Placed']
1.0    619
Name: value, dtype: int64
225449 ['Pericardiocentesis']
1.0    33
Name: value, dtype: int64
225430 ['Cardiac Cath']
1.0    633
Name: value, dtype: int64
225400 ['Bronchoscopy']
1.0    3027
Name: value, dtype: int64
225464 ['Cardioversion/Defibrillation']
1.0    695
Name: value, dtype: int64
225427 ['Angiography']
1.0    553
Name: value, dtype: int64
221217 ['Ultrasound']
1.0    7375
Name: value, dtype: int64
225451 ['Sputum Culture']
1.0    3770
Name: value, dtype: int64
225466 ['C

In [31]:
process_final_chart = final_chart.copy()

In [32]:
process_final_chart.shape

(233445, 4)

In [33]:
# #将0或者1的值进行处理
# process_final_chart.loc[(process_final_chart['itemid'].isin(cate_01)) & (~process_final_chart['value'].isin([0, 1])), 'value'] = 0

In [34]:
los=288
def create_Dict(chart):
    dataDic = {}
    feat = chart['itemid'].unique()  # 获取所有 itemid
    
    for hid in tqdm(chart.stay_id.unique()):  # 遍历每个 stay_id
        df2 = chart[chart['stay_id'] == hid]  # 获取对应 stay_id 的数据
        
        if df2.shape[0] == 0:
            print(f'空：{hid}')
            val = pd.DataFrame(np.zeros([los, len(feat)]), columns=feat)  # 如果没有数据，创建零填充的 DataFrame
            val = val.fillna(0)  # 确保没有 NaN

        else:
            # 按 start_time 和 itemid 对数据进行透视
            val = df2.pivot(index='start_time', columns='itemid', values='value')
            
            # 找到缺失的时间索引，补全缺失的时间点
            add_indices = pd.Index(range(los)).difference(val.index)
            add_df = pd.DataFrame(index=add_indices, columns=val.columns).fillna(0)  # 用零填充缺失的时间点
            val = pd.concat([val, add_df])  # 合并原数据和补全的数据
            val = val.sort_index()  # 按时间索引排序
            
            # 添加缺失的 itemid 列
            feat_df = pd.DataFrame(columns=list(set(feat) - set(val.columns)))  # 找到缺失的 itemid 列
            val = pd.concat([val, feat_df], axis=1)  # 将缺失的列添加到 val 中
            
            val = val[feat]  # 确保列的顺序与原始 feat 相同
            val = val.fillna(0)  # 填充 NaN 为 0

        # 保存数据到 CSV
        val.to_csv(f'D:/2024BI_Journal/MIMIC_IV/Final_Proc/{hid}.csv', index=False)

In [36]:
create_Dict(process_final_chart)

100%|████████████████████████████████████████████████████████████████████████████| 57475/57475 [28:18<00:00, 33.84it/s]


In [37]:
d_items[d_items.itemid.isin(process_final_chart.itemid)]#.itemid.values

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
272,221214,CT scan,CT scan,procedureevents,5-Imaging,None,Processes,NaN,NaN
273,221216,X-ray,X-ray,procedureevents,5-Imaging,None,Processes,NaN,NaN
274,221217,Ultrasound,Ultrasound,procedureevents,5-Imaging,None,Processes,NaN,NaN
276,221223,EEG,EEG,procedureevents,4-Procedures,None,Processes,NaN,NaN
322,223253,Magnetic Resonance Imaging,MRI,procedureevents,5-Imaging,None,Processes,NaN,NaN
598,224264,PICC Line,PICC Line,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN
600,224268,Trauma line,Trauma line,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN
602,224270,Dialysis Catheter,Dialysis Catheter,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN
603,224272,IABP line,IABP line,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN
604,224273,Presep Catheter,Presep Catheter,procedureevents,Access Lines - Invasive,None,Processes,NaN,NaN


In [38]:
def add_dynamic_process(dynamic, cate_01, str_col):
    # 将 0 替换为 NaN
    dynamic.replace(0, np.nan, inplace=True)
    
    # 删除所有值为 NaN 的列
    dynamic.dropna(axis=1, how='all', inplace=True)
    
    # 修改列名，如果列名在 cate_01 中，则加上前缀 'CHART_Cate_'
    # 如果列名在 str_col 中，则加上前缀 'CHART_Str_'
    # 其余的列加上前缀 'CHART_'
    dynamic.columns = [
        f"PROC_Cate_{col}" if int(col) in cate_01 else 
        f"PROC_Str_{col}" if int(col) in str_col else 
        f"PROC_{col}" 
        for col in dynamic.columns
    ]
    
    # 遍历列名并填补缺失值
    for col in dynamic.columns:
        # 如果列名前缀是 'CHART_' 或 'CHART_Str_'
        if col.startswith('PROC_Cate_'):
            # 如果是 'CHART_Cate_' 前缀的列，使用 0 填补缺失值
            dynamic[col] = dynamic[col].fillna(0)
        else:
            # 使用前向填充再进行后向填充
            dynamic[col] = dynamic[col].ffill().bfill()
    
    return dynamic

In [39]:
# 保存路径和初始化列名计数器
save_path = 'D:/2024BI_Journal/MIMIC_IV/Final_Proc_2/'

# 遍历每个 stay_id
for i in tqdm(os.listdir('D:/2024BI_Journal/MIMIC_IV/Final_Proc/')):
    try:
        # 读取 dynamic.csv 文件
        file_path = f'D:/2024BI_Journal/MIMIC_IV/Final_Proc/{i}'
        dynamic = pd.read_csv(file_path)
        dynamic = add_dynamic_process(dynamic, cate_01, str_col)
        #dynamic = dynamic.reindex(columns=final_all_ts_cols, fill_value=0)
        
        # 保存结果文件
        save_file_path = os.path.join(save_path, i)
        dynamic.to_csv(save_file_path, index=False)
    
    except Exception as e:
        # 打印错误信息并跳过该文件
        print(f"Error processing file {i}: {e}")

100%|████████████████████████████████████████████████████████████████████████████| 57475/57475 [47:59<00:00, 19.96it/s]


In [40]:
cols_count = Counter()

In [41]:
def median_ts_process(ts):
    # 将所有列值全部为 0 的列替换为 NaN
    ts.loc[:, (ts == 0).all(axis=0)] = np.nan  
    
    # 处理列数据
    processed_values = {}
    for col in ts.columns:
        if col.startswith('PROC_Cate_'):  
            # 对于 'CHART_Cate_' 开头的列，取最大值
            processed_values[col] = ts[col].max()
        elif col.startswith('PROC_Str_'):  
            # 对于 'CHART_Str_' 开头的列，取除 0 外出现次数最多的值
            non_zero_values = ts[col][ts[col] != '0']
            if not non_zero_values.empty:
                processed_values[col] = non_zero_values.mode().iloc[0]
            else:
                processed_values[col] = 0
        else:
            # 其他列取中位数
            processed_values[col] = ts[col].median()
    
    # 将结果转为 DataFrame
    result = pd.DataFrame([processed_values])
    
    return result

In [42]:
# 初始化空的 DataFrame 用于存储结果
all_ts = pd.DataFrame()

# 遍历每个 ICUSTAY_ID
for i in tqdm(os.listdir('D:/2024BI_Journal/MIMIC_IV/Final_Proc_2/')):
    # 读取 dynamic.csv 文件
    file_path = f'D:/2024BI_Journal/MIMIC_IV/Final_Proc_2/{i}'
    ts = pd.read_csv(file_path)
    
    # 调用 ts_process 函数进行数据处理
    ts = median_ts_process(ts)
    
    cols_count.update(ts.columns)
    
    ts['ICUSTAY_ID'] = i.split('.')[0]
    
    # 合并处理后的数据
    all_ts = pd.concat([all_ts, ts], ignore_index=True)
    
all_ts = all_ts.dropna(axis=1, how='all')

100%|████████████████████████████████████████████████████████████████████████████| 57475/57475 [21:27<00:00, 44.65it/s]


In [43]:
all_ts.shape

(57475, 54)

In [44]:
all_ts.head(2)

,PROC_Cate_225402,PROC_225752,PROC_Cate_221214,PROC_Cate_225433,PROC_Cate_227194,ICUSTAY_ID,PROC_225792,PROC_229351,PROC_Cate_225479,PROC_Cate_225401,...,PROC_Cate_225437,PROC_Cate_225475,PROC_224273,PROC_Cate_227713,PROC_Cate_225467,PROC_Cate_225449,PROC_Cate_225472,PROC_Cate_225450,PROC_Cate_225442,PROC_Cate_226237
0,1.0,1785.0,1.0,1.0,1.0,30000153,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,1.0,30000213,1614.0,1697.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
cols_count

Counter({'PROC_Cate_225402': 22932,
         'PROC_225752': 27918,
         'PROC_Cate_221214': 12029,
         'PROC_Cate_225433': 567,
         'PROC_Cate_227194': 16586,
         'PROC_225792': 26732,
         'PROC_229351': 15632,
         'PROC_Cate_225479': 291,
         'PROC_Cate_225401': 9555,
         'PROC_Cate_225966': 9235,
         'PROC_Cate_224385': 6318,
         'PROC_Cate_227712': 3906,
         'PROC_224264': 9014,
         'PROC_Cate_221216': 3484,
         'PROC_Cate_221223': 1722,
         'PROC_Cate_225439': 1673,
         'PROC_Cate_221217': 6546,
         'PROC_224270': 3154,
         'PROC_225441': 1648,
         'PROC_225802': 1456,
         'PROC_225805': 155,
         'PROC_225794': 1938,
         'PROC_Cate_225454': 6823,
         'PROC_Cate_225444': 992,
         'PROC_Cate_225440': 280,
         'PROC_Cate_225451': 3502,
         'PROC_224268': 605,
         'PROC_Cate_223253': 4713,
         'PROC_224272': 879,
         'PROC_Cate_225427': 532,
       

In [46]:
delete = ['PROC_Cate_226237']

In [47]:
keep_all_ts = all_ts.drop(columns=delete, errors='ignore')

In [48]:
keep_all_ts.shape

(57475, 53)

In [49]:
str_cols = []
cate_cols = []
c_cols = []
icuid = []

for col in keep_all_ts.columns:
    if col.startswith('PROC_Str_'):
        str_cols.append(col)  # 修正了变量名
    elif col.startswith('PROC_Cate_'):
        cate_cols.append(col)
    elif col=='ICUSTAY_ID':
        icuid.append(col)
    else:
        c_cols.append(col)


In [50]:
print(len(str_cols),len(cate_cols),len(c_cols))

0 40 12


In [51]:
feat = icuid + c_cols+cate_cols+str_cols
len(feat)

53

In [52]:
keep_all_ts = keep_all_ts[feat]

In [53]:
keep_all_ts.head()

,ICUSTAY_ID,PROC_225752,PROC_225792,PROC_229351,PROC_224264,PROC_224270,PROC_225441,PROC_225802,PROC_225805,PROC_225794,...,PROC_Cate_225448,PROC_Cate_228715,PROC_Cate_225437,PROC_Cate_225475,PROC_Cate_227713,PROC_Cate_225467,PROC_Cate_225449,PROC_Cate_225472,PROC_Cate_225450,PROC_Cate_225442
0,30000153,1785.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30000213,NaN,1614.0,1697.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,30000484,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,30000646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30001148,1055.0,1420.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
keep_all_ts.to_csv('D:/2024BI_Journal/MIMIC_IV/All_Proc_mean_max_mode_nan.csv',index=False)